# Handling Geojson file

Please run the scripts in the following order: 1.extracting_script -> 2.geo_handle -> 3.association_mining

## Import Libraries

In [10]:
import geopandas as gpd
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Import Datasets

In [11]:
lga_geo = gpd.read_file('LGA_2021_AUST_GDA94.geojson') #Import geo file
lga_geo.head()

,LGA_CODE21,LGA_NAME21,STE_CODE21,STE_NAME21,AUS_CODE21,AUS_NAME21,AREASQKM21,LOCI_URI21,SHAPE_Leng,SHAPE_Area,geometry
0,10050,Albury,1,New South Wales,AUS,Australia,305.6386,http://linked.data.gov.au/dataset/asgsed3/LGA2...,1.321768,0.030560,"MULTIPOLYGON (((146.86565 -36.07293, 146.86511..."
1,10180,Armidale Regional,1,New South Wales,AUS,Australia,7809.4406,http://linked.data.gov.au/dataset/asgsed3/LGA2...,6.034583,0.732825,"MULTIPOLYGON (((151.32425 -30.26923, 151.32419..."
2,10250,Ballina,1,New South Wales,AUS,Australia,484.9692,http://linked.data.gov.au/dataset/asgsed3/LGA2...,1.511121,0.044843,"MULTIPOLYGON (((153.57106 -28.87382, 153.57105..."
3,10300,Balranald,1,New South Wales,AUS,Australia,21690.7493,http://linked.data.gov.au/dataset/asgsed3/LGA2...,11.489913,2.115528,"MULTIPOLYGON (((143.00432 -33.78165, 143.01538..."
4,10470,Bathurst Regional,1,New South Wales,AUS,Australia,3817.8645,http://linked.data.gov.au/dataset/asgsed3/LGA2...,5.395114,0.370149,"MULTIPOLYGON (((149.91212 -33.39582, 149.91146..."


In [12]:
lga_geo.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 566 entries, 0 to 565
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   LGA_CODE21  566 non-null    object  
 1   LGA_NAME21  566 non-null    object  
 2   STE_CODE21  566 non-null    object  
 3   STE_NAME21  566 non-null    object  
 4   AUS_CODE21  566 non-null    object  
 5   AUS_NAME21  566 non-null    object  
 6   AREASQKM21  566 non-null    float64 
 7   LOCI_URI21  566 non-null    object  
 8   SHAPE_Leng  566 non-null    float64 
 9   SHAPE_Area  566 non-null    float64 
 10  geometry    566 non-null    geometry
dtypes: float64(3), geometry(1), object(7)
memory usage: 48.8+ KB


In [13]:
location_dimension = pd.read_csv('location_dimension.csv') #Import location dimension
# Drop the column only if it exists
if 'LGA_NAME21' in location_dimension.columns: #Remove column if existed to avoid crash
    location_dimension.drop(columns=['LGA_NAME21'], inplace=True)
location_dimension.head()

,LocationKey,State,LGA,LGACode
0,1,NSW,Wagga Wagga,17750.0
1,2,NSW,Hawkesbury,13800.0
2,3,Tas,Northern Midlands,64610.0
3,4,NSW,Armidale,10180.0
4,5,Qld,Lockyer Valley,34580.0


In [14]:
location_dimension.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 555 entries, 0 to 554
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   LocationKey  555 non-null    int64  
 1   State        517 non-null    object 
 2   LGA          555 non-null    object 
 3   LGACode      547 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 17.5+ KB


In [15]:
locaion_temp = location_dimension[['LGA','LGACode']] #Create temporary dataset

In [16]:
locaion_temp['LGACode'] = locaion_temp['LGACode'].fillna(0).astype(int).astype(str)  # Replace NaN with 0
geo_check = pd.merge(
    locaion_temp[['LGA', 'LGACode']],
    lga_geo,
    left_on='LGACode',
    right_on='LGA_CODE21',
    how='left'
)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_36880\1368445059.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  locaion_temp['LGACode'] = locaion_temp['LGACode'].fillna(0).astype(int).astype(str)  # Replace NaN with 0


In [17]:
geo_check[geo_check['LGA']!= geo_check['LGA_NAME21']][['LGA','LGACode','LGA_NAME21','LGA_CODE21']] #Identifying the mismatched 

,LGA,LGACode,LGA_NAME21,LGA_CODE21
3,Armidale,10180,Armidale Regional,10180
6,Unidentified_WA,0,NaN,NaN
12,Upper Lachlan,17640,Upper Lachlan Shire,17640
17,Upper Hunter,17620,Upper Hunter Shire,17620
19,Mid-Western,15270,Mid-Western Regional,15270
21,Greater Hume,13340,Greater Hume Shire,13340
38,Tamworth,17310,Tamworth Regional,17310
65,The Hills,17420,The Hills Shire,17420
87,Queanbeyan-Palerang,16490,Queanbeyan-Palerang Regional,16490
88,Snowy Monaro,17040,Snowy Monaro Regional,17040


In [18]:
geo_check.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 555 entries, 0 to 554
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   LGA         555 non-null    object  
 1   LGACode     555 non-null    object  
 2   LGA_CODE21  546 non-null    object  
 3   LGA_NAME21  546 non-null    object  
 4   STE_CODE21  546 non-null    object  
 5   STE_NAME21  546 non-null    object  
 6   AUS_CODE21  546 non-null    object  
 7   AUS_NAME21  546 non-null    object  
 8   AREASQKM21  546 non-null    float64 
 9   LOCI_URI21  546 non-null    object  
 10  SHAPE_Leng  546 non-null    float64 
 11  SHAPE_Area  546 non-null    float64 
 12  geometry    546 non-null    geometry
dtypes: float64(3), geometry(1), object(9)
memory usage: 56.5+ KB


In [19]:
new_location = pd.concat([location_dimension, geo_check[['LGA_NAME21','LGA_CODE21']]], axis=1).reindex(location_dimension.index) #Join file to introduce new LGA_NAME21 column
new_location

,LocationKey,State,LGA,LGACode,LGA_NAME21,LGA_CODE21
0,1,NSW,Wagga Wagga,17750.0,Wagga Wagga,17750
1,2,NSW,Hawkesbury,13800.0,Hawkesbury,13800
2,3,Tas,Northern Midlands,64610.0,Northern Midlands,64610
3,4,NSW,Armidale,10180.0,Armidale Regional,10180
4,5,Qld,Lockyer Valley,34580.0,Lockyer Valley,34580
...,...,...,...,...,...,...
550,551,NaN,Yalgoo,59350.0,Yalgoo,59350
551,552,NaN,Belyuen,70540.0,Belyuen,70540
552,553,NaN,Darwin Waterfront Precinct,71150.0,Darwin Waterfront Precinct,71150
553,554,NaN,Tiwi Islands,74050.0,Tiwi Islands,74050


In [20]:
new_location[(new_location['LGA']!=new_location['LGA_NAME21'])&(new_location['LGA_CODE21']!=new_location['LGACode'])] #Final check

,LocationKey,State,LGA,LGACode,LGA_NAME21,LGA_CODE21
3,4,NSW,Armidale,10180.0,Armidale Regional,10180
6,7,WA,Unidentified_WA,NaN,NaN,NaN
12,13,NSW,Upper Lachlan,17640.0,Upper Lachlan Shire,17640
17,18,NSW,Upper Hunter,17620.0,Upper Hunter Shire,17620
19,20,NSW,Mid-Western,15270.0,Mid-Western Regional,15270
21,22,NSW,Greater Hume,13340.0,Greater Hume Shire,13340
38,39,NSW,Tamworth,17310.0,Tamworth Regional,17310
65,66,NSW,The Hills,17420.0,The Hills Shire,17420
87,88,NSW,Queanbeyan-Palerang,16490.0,Queanbeyan-Palerang Regional,16490
88,89,NSW,Snowy Monaro,17040.0,Snowy Monaro Regional,17040


In [21]:
new_location.loc[new_location['LGA'] == 'Merri-bek', 'LGA_NAME21'] = 'Moreland' #Ensure Merri-bek keep its old name in LGA_NAME21
new_location['LGACode'] = new_location['LGACode'].fillna(0).astype(int).astype(str)
new_location['LGACode'] = np.where(new_location['LGACode'] == '0', 'NonExist', new_location['LGACode']) #Convert 0 to NonExist to avoid missing data
new_location['LGA_NAME21'] = np.where(new_location['LGA_NAME21'].isna(), new_location['LGA'], new_location['LGA_NAME21'])
if 'LGA_CODE21' in new_location.columns:
    new_location.drop(columns=['LGA_CODE21'], inplace=True)
new_location.to_csv('location_dimension.csv', index=False)